In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0' 
 
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
 
# ============================================================
# PATHS — sesuaikan dengan struktur folder kamu
# ============================================================
TRAIN_CSV       = 'youtube_enriched_cluster.csv'
NEW_CSV         = 'new_videos.csv'
TRAIN_THUMB_DIR = 'thumbnails'
NEW_THUMB_DIR   = 'thumbnails_new'
OUTPUT_CSV      = 'new_videos_labeled.csv'
MODELS_DIR      = 'saved_models'
# ============================================================
 
os.makedirs(MODELS_DIR, exist_ok=True)

In [2]:
METADATA_COLS = [
    'view_rate', 'like_count', 'comment_count', 'video_age_days',
    'like_rate', 'comment_rate', 'engagement_rate',
    'subscriber_count', 'log_view_count', 'log_like_count',
    'log_comment_count', 'log_subscriber_count'
]
 
# --- Load CNN model ---
print("🔧 Loading EfficientNetB0...")
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
gap_output = GlobalAveragePooling2D()(base_model.output)
cnn_model  = Model(inputs=base_model.input, outputs=gap_output)
 
 
def extract_cnn_features(df, thumb_dir, id_col='thumbnail_path'):
    images, valid_idx = [], []
    for i, row in df.iterrows():
        vid_id = os.path.splitext(os.path.basename(str(row.get(id_col, ''))))[0]
        path   = os.path.join(thumb_dir, f'{vid_id}.jpg')
        if not os.path.exists(path):
            path = os.path.join(thumb_dir, f'_{vid_id}.jpg')
        if not os.path.exists(path):
            print(f"   ⚠️  Not found: {vid_id}, skipping row {i}")
            continue
        img = load_img(path, target_size=(224, 224))
        images.append(preprocess_input(img_to_array(img)))
        valid_idx.append(i)
    if not images:
        raise FileNotFoundError(f"No thumbnails found in '{thumb_dir}'")
    features = cnn_model.predict(np.array(images), verbose=1, batch_size=16)
    return features, valid_idx

🔧 Loading EfficientNetB0...


In [3]:
# ============================================================
# STEP 1 — Re-fit Scaler, OHE, KMeans dari training data
# ============================================================
print("\n📂 Loading training data...")
train_df = pd.read_csv(TRAIN_CSV)
 
# Pastikan semua fitur ada
train_df['view_rate']       = train_df['view_count'] / train_df['video_age_days']
train_df['like_rate']       = train_df['like_count'] / train_df['view_count']
train_df['comment_rate']    = train_df['comment_count'] / train_df['view_count']
train_df['engagement_rate'] = (train_df['like_count'] + train_df['comment_count']) / train_df['view_count']
train_df['log_view_count']       = np.log1p(train_df['view_count'])
train_df['log_like_count']       = np.log1p(train_df['like_count'])
train_df['log_comment_count']    = np.log1p(train_df['comment_count'])
train_df['log_subscriber_count'] = np.log1p(train_df['subscriber_count'])
train_df.fillna(0, inplace=True)
 
print("🖼️  Extracting CNN features from training thumbnails...")
X_cnn_train, train_valid_idx = extract_cnn_features(train_df, TRAIN_THUMB_DIR)
train_df = train_df.loc[train_valid_idx].reset_index(drop=True)
 
print("⚙️  Fitting OHE + Scaler + KMeans...")
ohe       = OneHotEncoder(sparse=False, handle_unknown='ignore')
cat_train = ohe.fit_transform(train_df[['Category']])
 
X_meta_train  = np.concatenate([train_df[METADATA_COLS].values, cat_train], axis=1)
scaler        = StandardScaler()
X_meta_scaled = scaler.fit_transform(X_meta_train)
X_all_train   = np.concatenate([X_cnn_train, X_meta_scaled], axis=1)
 
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X_all_train)
 
joblib.dump(scaler, f'{MODELS_DIR}/scaler.pkl')
joblib.dump(kmeans, f'{MODELS_DIR}/kmeans.pkl')
joblib.dump(ohe,    f'{MODELS_DIR}/ohe.pkl')
print(f"✅ Models saved to '{MODELS_DIR}/'")
 
train_df['cluster_label_refit'] = kmeans.labels_
print("\n📊 Cluster means (refit) — harus match training asli:")
print(train_df.groupby('cluster_label_refit')[['view_rate', 'like_rate', 'engagement_rate']]
      .mean().sort_values('view_rate', ascending=False))


📂 Loading training data...
🖼️  Extracting CNN features from training thumbnails...
65/65 [==============================] - 5s 21ms/step
⚙️  Fitting OHE + Scaler + KMeans...


C:\Users\LENOVO\anaconda3\envs\tf-gpu-clean\lib\site-packages\sklearn\preprocessing\_encoders.py:975: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(
C:\Users\LENOVO\anaconda3\envs\tf-gpu-clean\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


✅ Models saved to 'saved_models/'

📊 Cluster means (refit) — harus match training asli:
                        view_rate  like_rate  engagement_rate
cluster_label_refit                                          
2                    12016.853265   0.029290         0.031104
1                     5251.256054   0.024102         0.028496
0                     3833.517541   0.046249         0.047751


In [4]:
# ============================================================
# STEP 2 — Hitung fitur untuk new data
# ============================================================
print("\n📂 Loading new videos...")
new_df = pd.read_csv(NEW_CSV)
 
# Hitung video_age_days dari published_at
new_df['published_at']  = pd.to_datetime(new_df['published_at'], utc=True)
new_df['video_age_days'] = (pd.Timestamp.utcnow() - new_df['published_at']).dt.days
new_df['video_age_days'] = new_df['video_age_days'].replace(0, 1)  # hindari division by zero
 
# Hitung semua fitur dengan rumus yang sama persis dengan training
new_df['view_rate']       = new_df['view_count'] / new_df['video_age_days']
new_df['like_rate']       = new_df['like_count'] / new_df['view_count'].replace(0, np.nan)
new_df['comment_rate']    = new_df['comment_count'] / new_df['view_count'].replace(0, np.nan)
new_df['engagement_rate'] = (new_df['like_count'] + new_df['comment_count']) / new_df['view_count'].replace(0, np.nan)
new_df['log_view_count']       = np.log1p(new_df['view_count'])
new_df['log_like_count']       = np.log1p(new_df['like_count'])
new_df['log_comment_count']    = np.log1p(new_df['comment_count'])
new_df['log_subscriber_count'] = np.log1p(new_df['subscriber_count'])
new_df.fillna(0, inplace=True)
 
# Rename category → Category agar cocok dengan OHE
new_df.rename(columns={'category': 'Category'}, inplace=True)


📂 Loading new videos...


In [5]:
# ============================================================
# STEP 3 — Extract CNN features dari thumbnail baru
# ============================================================
print("🖼️  Extracting CNN features from new thumbnails...")
X_cnn_new, new_valid_idx = extract_cnn_features(new_df, NEW_THUMB_DIR, id_col='thumbnail_path')
new_df = new_df.loc[new_valid_idx].reset_index(drop=True)

🖼️  Extracting CNN features from new thumbnails...
101/101 [==============================] - 2s 21ms/step


In [6]:
# ============================================================
# STEP 4 — Assign cluster labels
# ============================================================
print("🔮 Assigning cluster labels...")
cat_new      = ohe.transform(new_df[['Category']])
X_meta_new   = np.concatenate([new_df[METADATA_COLS].values, cat_new], axis=1)
X_meta_new_s = scaler.transform(X_meta_new)
X_all_new    = np.concatenate([X_cnn_new, X_meta_new_s], axis=1)
 
new_df['cluster_label'] = kmeans.predict(X_all_new)

🔮 Assigning cluster labels...


In [7]:
print(train_df[['view_count', 'video_age_days', 'view_rate']].head())
print((train_df['view_count'] / train_df['video_age_days']).head())

   view_count  video_age_days    view_rate
0      422674             313  1350.396166
1       74080              82   903.414634
2      580663             270  2150.603704
3      544136              80  6801.700000
4      110290              53  2080.943396
0    1350.396166
1     903.414634
2    2150.603704
3    6801.700000
4    2080.943396
dtype: float64


In [8]:
# ============================================================
# STEP 5 — Simpan output
# ============================================================
new_df.to_csv(OUTPUT_CSV, index=False)
print(f"\n✅ Done! {len(new_df)} videos labeled → '{OUTPUT_CSV}'")
print("\n📊 Cluster distribution:")
print(new_df['cluster_label'].value_counts().sort_index())
print("\n📊 Cluster means (new data) — bandingkan dengan training:")
print(new_df.groupby('cluster_label')[['view_rate', 'like_rate', 'engagement_rate']]
      .mean().sort_values('view_rate', ascending=False))


✅ Done! 1609 videos labeled → 'new_videos_labeled.csv'

📊 Cluster distribution:
cluster_label
0    953
1    531
2    125
Name: count, dtype: int64

📊 Cluster means (new data) — bandingkan dengan training:
                  view_rate  like_rate  engagement_rate
cluster_label                                          
2              49540.255136   0.024083         0.025438
1               7476.745236   0.026464         0.031477
0               4143.437212   0.034926         0.036425
